# Fiche 1 — Du tableur à la table : ligne, entité, clé primaire

| | |
|---|---|
| 🎯 **Objectif** | Savoir dire ce que représente une ligne et pourquoi une table a besoin d'une clé primaire |
| ⏱️ **Durée** | 30 min |
| 🧰 **Prérequis** | Aucun : c'est la première fiche |
| 🔗 **Dans le projet** | Partie 1, étape 1 (comprendre les données) |

> **L'idée clé.** Une table ressemble à une feuille de tableur, avec une règle en plus : chaque ligne doit pouvoir être désignée sans ambiguïté. C'est le rôle de la **clé primaire**.

> **Comment utiliser les fiches.** Les cellules grises exécutent du SQL ; le même SQL est donné dans un encadré
> « À coller dans SQLite Studio » pour que tu refasses chaque manipulation dans ton outil.

## 1. Une ligne = une chose du monde réel

Dans le fichier des valeurs foncières, chaque ligne décrit **une vente** d'un bien. Ce « type de chose » s'appelle
une **entité** : la vente, le bien, la commune, la région sont 4 entités, qui deviendront 4 tables.

In [1]:
import sqlite3

import pandas as pd

# Une base « jetable » en mémoire : on peut tout casser, rien n'est écrit sur le disque.
# isolation_level=None : chaque instruction est validée aussitôt, comme dans SQLite Studio.
base = sqlite3.connect(":memory:", isolation_level=None)


def sql(requete: str) -> pd.DataFrame | None:
    """Exécute du SQL (une ou plusieurs instructions) et affiche le résultat s'il y en a un."""
    if requete.strip().rstrip(";").count(";"):
        base.executescript(requete)
        return None
    curseur = base.execute(requete)
    if curseur.description:
        return pd.DataFrame(curseur.fetchall(), columns=[c[0] for c in curseur.description])
    return None

sql("""
CREATE TABLE vente_sans_cle (
    date_vente      TEXT,
    valeur_fonciere REAL,
    commune         TEXT
);
""")

## 2. ⚠️ Le piège : une table sans clé accepte les doublons

In [2]:
sql("INSERT INTO vente_sans_cle VALUES ('2020-01-02', 165000, 'Chevry')")
sql("INSERT INTO vente_sans_cle VALUES ('2020-01-02', 165000, 'Chevry')")  # la même vente, saisie deux fois
sql("SELECT * FROM vente_sans_cle")

,date_vente,valeur_fonciere,commune
0,2020-01-02,165000.0,Chevry
1,2020-01-02,165000.0,Chevry


À coller dans SQLite Studio :
```sql
CREATE TABLE vente_sans_cle (date_vente TEXT, valeur_fonciere REAL, commune TEXT);
INSERT INTO vente_sans_cle VALUES ('2020-01-02', 165000, 'Chevry');
INSERT INTO vente_sans_cle VALUES ('2020-01-02', 165000, 'Chevry');
SELECT * FROM vente_sans_cle;
```

Deux lignes identiques : une vente comptée deux fois, ou deux ventes réelles au même prix le même jour ?
**Impossible de le savoir.** Tous les comptages seraient faussés.

## 3. La clé primaire : un identifiant unique et jamais vide

In [3]:
sql("""
CREATE TABLE vente (
    id_vente        INTEGER PRIMARY KEY,   -- identifiant unique de chaque vente
    date_vente      TEXT NOT NULL,
    valeur_fonciere REAL
);
""")
sql("INSERT INTO vente VALUES (1, '2020-01-02', 165000)")
try:
    sql("INSERT INTO vente VALUES (1, '2020-01-02', 165000)")  # même identifiant : refusé
except sqlite3.IntegrityError as erreur:
    print("SQLite refuse :", erreur)

SQLite refuse : UNIQUE constraint failed: vente.id_vente


| Type de clé | Exemple dans le projet | Quand l'utiliser |
|---|---|---|
| **Numéro attribué** (auto-incrément) | `id_vente`, `id_bien` : 1, 2, 3… | aucune donnée naturelle n'est unique |
| **Clé naturelle** | `id_region` = code INSEE de la région | une donnée existante est déjà unique et stable |
| **Clé concaténée** | `id_codedep_codecommune` = « 34 » + « 172 » | aucune colonne seule n'est unique, leur combinaison l'est |

### ✍️ À toi de jouer

Le **nom de la commune** pourrait-il servir de clé primaire d'une table des communes ? Vérifie-le dans ton tableur sur le fichier des valeurs foncières : crée un tableau croisé dynamique avec *Commune* en lignes et le **nombre distinct** de *Code departement* en valeurs, puis filtre les valeurs supérieures à 1.

**Ta réponse** (quelques phrases) : …

<details>
<summary><b>💡 Voir une solution</b> (après avoir cherché !)</summary>

```text
Commune | Nombre distinct de Code departement
…       | 2
…       | 2
```

Plusieurs communes portent le même nom dans des départements différents (des **homonymes**) : le nom n'est pas unique, ce n'est pas une clé. D'où la clé code département + code commune.
</details>

## ✅ À retenir

1. Une table = une entité ; une ligne = une occurrence (une vente, une commune…).
2. Sans clé primaire, une base accepte des doublons qu'on ne peut plus distinguer.
3. Une clé primaire est **unique** et **jamais vide** ; la base le vérifie à chaque insertion.
4. Un nom n'est pas une bonne clé : il peut être partagé (homonymes) ou changer.